In [1]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
import numpy as np
from pathlib import Path
import json
import pandas as pd
from tfmbench.datasets import BaseTabularDataset as TabularDataset
from tfmbench.evaluate import evaluate
from tfmbench.benchmark import benchmark_models
from tfmbench.datasets.talent import load_talent_dataset
from tfmbench.utils import get_tab_split

In [2]:
TABPFN_TOKEN="tabpfn_sk_oIa-pBgg3FDBGrM95bfSAh-18ypWp8FCvhQ_Bzg3nec"
import torch
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps") 
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

Using device: mps


In [3]:
from pathlib import Path
import ast
import struct
import json

import numpy as np
import pandas as pd


def get_npy_info(path):
    """
    Read shape and dtype from a .npy file without loading it.
    """
    path = Path(path)

    with open(path, "rb") as f:

        magic = f.read(6)

        if magic != b"\x93NUMPY":
            raise ValueError(f"{path} is not a valid .npy file")

        major = int.from_bytes(f.read(1), "little")
        minor = int.from_bytes(f.read(1), "little")

        if major == 1:
            header_len = struct.unpack("<H", f.read(2))[0]
            encoding = "latin1"

        elif major in (2, 3):
            header_len = struct.unpack("<I", f.read(4))[0]
            encoding = "utf-8" if major == 3 else "latin1"

        else:
            raise ValueError(
                f"Unsupported .npy version {major}.{minor}"
            )

        header = f.read(header_len).decode(encoding)

        header_dict = ast.literal_eval(header)

        shape = tuple(header_dict["shape"])
        dtype = np.dtype(header_dict["descr"])

    return shape, dtype


def get_talent_dataset_stats(root):

    root = Path(root)
    rows = []

    for dataset_dir in sorted(root.iterdir()):

        if not dataset_dir.is_dir():
            continue

        if not (dataset_dir / "y_train.npy").exists():
            continue

        try:
            
            info_path = dataset_dir / "info.json"

            if info_path.exists():
                with open(info_path, "r") as f:
                    info = json.load(f)
            else:
                info = {}

            task_type = info.get("task_type", "unknown")

            split_sizes = {}

            for split in ["train", "val", "test"]:

                y_path = dataset_dir / f"y_{split}.npy"

                if y_path.exists():

                    shape, _ = get_npy_info(y_path)

                    split_sizes[split] = shape[0]

                else:

                    split_sizes[split] = 0


            n_train = split_sizes["train"]
            n_val = split_sizes["val"]
            n_test = split_sizes["test"]

            total_rows = (
                n_train
                + n_val
                + n_test
            )

            n_num_features = 0
            n_cat_features = 0

            n_path = dataset_dir / "N_train.npy"
            c_path = dataset_dir / "C_train.npy"

            if n_path.exists():

                shape, _ = get_npy_info(n_path)

                if len(shape) > 1:
                    n_num_features = shape[1]

            if c_path.exists():

                shape, _ = get_npy_info(c_path)

                if len(shape) > 1:
                    n_cat_features = shape[1]

            n_features = (
                n_num_features
                + n_cat_features
            )

            y_train = np.load(
                dataset_dir / "y_train.npy",
                allow_pickle=True,
            )

            y_train = np.asarray(
                y_train
            ).reshape(-1)

            is_regression = (
                "regression"
                in str(task_type).lower()
            )

            if is_regression:

                n_classes = None
                class_counts = None

                y_numeric = y_train.astype(float)

                target_mean = float(
                    np.mean(y_numeric)
                )

                target_std = float(
                    np.std(y_numeric)
                )

            else:

                unique_values, counts = np.unique(
                    y_train,
                    return_counts=True,
                )

                n_classes = len(unique_values)

                class_counts = {
                    str(cls): int(count)
                    for cls, count in zip(
                        unique_values,
                        counts,
                    )
                }

                target_mean = None
                target_std = None

            train_fraction = (
                n_train / total_rows
                if total_rows
                else None
            )

            rows_per_feature = (
                total_rows / n_features
                if n_features
                else None
            )


            rows.append({
                "dataset": dataset_dir.name,
                "task": task_type,

                "total_rows": total_rows,
                "train_rows": n_train,
                "val_rows": n_val,
                "test_rows": n_test,

                "n_features": n_features,
                "n_numeric_features": n_num_features,
                "n_categorical_features": n_cat_features,

                "n_classes": n_classes,
                #"class_counts_train": class_counts,
                #"target_mean": target_mean,
                #"target_std": target_std,
            })

        except Exception as e:

            print(
                f"Failed to process {dataset_dir.name}: "
                f"{type(e).__name__}: {e}"
            )

    df = pd.DataFrame(rows)

    if not df.empty:

        df = (
            df
            .sort_values(
                "total_rows",
                ascending=False,
            )
            .reset_index(drop=True)
        )

    return df

In [4]:
stats_df = get_talent_dataset_stats(
    "./data/datasets-talent-highdim-cv"
)

stats_df

,dataset,task,total_rows,train_rows,val_rows,test_rows,n_features,n_numeric_features,n_categorical_features,n_classes
0,gisette_1,binclass,7000,4480,1120,1400,5000,5000,0,2
1,gisette_5,binclass,7000,4480,1120,1400,5000,5000,0,2
2,gisette_4,binclass,7000,4480,1120,1400,5000,5000,0,2
3,gisette_3,binclass,7000,4480,1120,1400,5000,5000,0,2
4,gisette_2,binclass,7000,4480,1120,1400,5000,5000,0,2
...,...,...,...,...,...,...,...,...,...,...
125,GLIOMA_5,multiclass,50,32,8,10,4434,4434,0,4
126,GLIOMA_4,multiclass,50,32,8,10,4434,4434,0,4
127,GLIOMA_3,multiclass,50,32,8,10,4434,4434,0,4
128,GLIOMA_2,multiclass,50,32,8,10,4434,4434,0,4


In [5]:
stats_df = get_talent_dataset_stats(
    "./data/datasets-talent-large"
)

stats_df

,dataset,task,total_rows,train_rows,val_rows,test_rows,n_features,n_numeric_features,n_categorical_features,n_classes
0,Airlines_DepDelay_10M,regression,10000000,6400000,1600000,2000000,9,6,3,NaN
1,KDDCup99,multiclass,4898431,3134995,783749,979687,41,32,9,23.0
2,sf-police-incidents,binclass,2215023,1417614,354404,443005,8,3,5,2.0
3,microsoft,regression,1200192,723412,235259,241521,136,136,0,NaN
4,poker-hand,multiclass,1025009,656005,164002,205002,10,10,0,10.0
5,Higgs,binclass,1000000,640000,160000,200000,28,28,0,2.0
6,BNG(credit-a),binclass,1000000,640000,160000,200000,15,6,9,2.0
7,Smoking_and_Drinking_Dataset_with_body_signal,binclass,991346,634460,158616,198270,23,22,1,2.0
8,yahoo,regression,709877,473134,71083,165660,699,699,0,NaN
9,Data_Science_for_Good_Kiva_Crowdfunding,multiclass,671205,429571,107393,134241,11,7,4,4.0


In [ ]:
import gc
import pandas as pd
import torch

from tfmbench.evaluate import evaluate
from tfmbench.datasets.talent import load_talent_dataset


def _cleanup_memory():
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        try:
            torch.mps.empty_cache()
        except Exception:
            pass


def benchmark_dataset_model_compatibility(
    stats_df,
    root,
    models,
    device,
    tabpfn_token=None,
    model_kwargs=None,
):

    model_kwargs = model_kwargs or {}

    rows = []
    errors = []

    for _, stats in stats_df.iterrows():

        dataset_name = stats["dataset"]

        print("\n" + "=" * 100)
        print(
            f"{dataset_name} | "
            f"N={int(stats['total_rows']):,} | "
            f"N_train={int(stats['train_rows']):,} | "
            f"F={int(stats['n_features'])}"
        )
        print("=" * 100)

        row = {
            "dataset": dataset_name,
            "task": stats["task"],
            "N": int(stats["total_rows"]),
            "N_train": int(stats["train_rows"]),
            "F": int(stats["n_features"]),
        }

        try:
            data = load_talent_dataset(
                dataset_name,
                root=root,
                include_val=False,
            )

        except Exception as e:
            print(f"  DATASET LOAD FAILED: {e}")

            for model_name in models:
                row[model_name] = "-"

                errors.append({
                    "dataset": dataset_name,
                    "model": model_name,
                    "stage": "dataset_loading",
                    "error_type": type(e).__name__,
                    "error": str(e),
                })

            rows.append(row)
            continue

        for model_name in models:

            print(f"  Running {model_name}...", end=" ")

            kwargs = model_kwargs.get(
                model_name,
                {},
            )

            try:
                result = evaluate(
                    model_name=model_name,
                    data=data,
                    device=device,
                    tabpfn_token=tabpfn_token,
                    model_kwargs=kwargs,
                    return_predictions=False,
                )

                row[model_name] = "*"

                print("✓")

                del result

            except Exception as e:
                row[model_name] = "-"

                print(
                    f"FAILED: "
                    f"{type(e).__name__}: {e}"
                )

                errors.append({
                    "dataset": dataset_name,
                    "model": model_name,
                    "stage": "evaluation",
                    "error_type": type(e).__name__,
                    "error": str(e),
                })

            finally:
                _cleanup_memory()

        rows.append(row)

        del data
        _cleanup_memory()

    compatibility_df = pd.DataFrame(rows)

    errors_df = pd.DataFrame(errors)

    return compatibility_df, errors_df

In [9]:
MODELS = [
    "tabicl_v1",
    "tabicl_v1.1",
    "tabicl_v2",

    "tabpfn_v2",
    "tabpfn_v2.5",
    "tabpfn_v2.6",
    "tabpfn_v3",
    "tabpfn_v3.5",
    "tabpfn_v3.5_fast",

    "tabdpt_v1.3",
]


MODEL_KWARGS = {
    "xgboost": {
        "n_estimators": 500,
        "max_depth": 8,
    },

    "catboost": {
        "iterations": 500,
        "depth": 8,
    },

    "tabdpt_v1.3": {
        "context_size": 8192,
        "n_ensembles": 1,
    },

    "tabpfn_v3": {
        "n_estimators": 8,
    },
}

In [10]:
compatibility_df, errors_df = (
    benchmark_dataset_model_compatibility(
        stats_df=stats_df,
        root="./data/datasets-talent-large",
        models=MODELS,
        device=device,
        tabpfn_token=TABPFN_TOKEN,
        model_kwargs=MODEL_KWARGS,
    )
)


Airlines_DepDelay_10M | N=10,000,000 | N_train=6,400,000 | F=9
  Running tabicl_v1... FAILED: ValueError: Regression: v2
  Running tabicl_v1.1... FAILED: ValueError: Regression: v2
  Running tabicl_v2... 

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/tabicl/_sklearn/preprocessing.py:138: UserWarning: The following categorical columns have a cardinality above 40: ['num_3', 'num_4', 'num_5', 'cat_1', 'cat_2']. High-cardinality columns might benefit from a better encoding than ordinal encoding, e.g. Skrub's TableVectorizer for strings.
  warnings.warn(


FAILED: RuntimeError: MPS backend out of memory (MPS allocated: 5.74 GiB, other allocations: 1.69 MiB, max allowed: 8.29 GiB). Tried to allocate 3.44 GiB on shared pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).
  Running tabpfn_v2... FAILED: TabPFNValidationError: Number of samples `6,400,000` in the input data is greater than the maximum number of samples `10,000` officially supported by TabPFN. Set `ignore_pretraining_limits=True` to override this error!
  Running tabpfn_v2.5... FAILED: TabPFNValidationError: Number of samples `6,400,000` in the input data is greater than the maximum number of samples `50,000` officially supported by TabPFN. Set `ignore_pretraining_limits=True` to override this error!
  Running tabpfn_v2.6... FAILED: TabPFNValidationError: Number of samples `6,400,000` in the input data is greater than the maximum number of samples `100,000` officially supported by TabPFN. Set `ignore_pretraini

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/tabicl/_sklearn/preprocessing.py:138: UserWarning: The following categorical columns have a cardinality above 40: ['num_0', 'num_1', 'num_2', 'num_7', 'num_8', 'num_13', 'num_14', 'num_15', 'num_16', 'num_17', 'num_18', 'num_19', 'num_20', 'num_21', 'num_22', 'num_23', 'num_24', 'num_25', 'num_26', 'num_27', 'num_28', 'num_29', 'num_30', 'num_31', 'cat_1']. High-cardinality columns might benefit from a better encoding than ordinal encoding, e.g. Skrub's TableVectorizer for strings.
  warnings.warn(


FAILED: RuntimeError: MPS backend out of memory (MPS allocated: 7.59 GiB, other allocations: 1.69 MiB, max allowed: 8.29 GiB). Tried to allocate 1.97 GiB on shared pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).
  Running tabicl_v1.1... 

...

OOM: halving col_chunk_size to 2
OOM: halving col_chunk_size to 1


FAILED: TabPFNMPSOutOfMemoryError: MPS out of memory with 198,270 test samples.

This is issue is usually caused by one of the following two reasons:

1) Large test set — split into batches:

    predictions = []
    for i in range(0, len(X_test), 100):
        pred = model.predict_proba(X_test[i:i + 100])
        predictions.append(pred)
    predictions = np.vstack(predictions)

   Only with fit_mode='fit_with_cache', test rows are chunked
   automatically — lower the TABPFN_MAX_BATCHED_TEST_ROWS
   environment variable (currently 32768) to
   reduce peak memory without changing your code.

2) Large training set — batching won't help.
   Subsample your training data; see https://docs.priorlabs.ai
   for further guidance.

Your sizes: 634,460 train / 198,270 test samples, 23 features.
Not sure which? If model.predict_proba(X_test[:1]) also fails, it's (2).

Original error: MPS backend out of memory (MPS allocated: 5.27 GiB, other allocations: 36.91 MiB, max allowed: 8.29 GiB). Tried to

OOM: halving col_chunk_size to 2


FAILED: TabPFNMPSOutOfMemoryError: MPS out of memory with 198,270 test samples.

This is issue is usually caused by one of the following two reasons:

1) Large test set — split into batches:

    predictions = []
    for i in range(0, len(X_test), 100):
        pred = model.predict_proba(X_test[i:i + 100])
        predictions.append(pred)
    predictions = np.vstack(predictions)

   Only with fit_mode='fit_with_cache', test rows are chunked
   automatically — lower the TABPFN_MAX_BATCHED_TEST_ROWS
   environment variable (currently 32768) to
   reduce peak memory without changing your code.

2) Large training set — batching won't help.
   Subsample your training data; see https://docs.priorlabs.ai
   for further guidance.

Your sizes: 634,460 train / 198,270 test samples, 23 features.
Not sure which? If model.predict_proba(X_test[:1]) also fails, it's (2).

Original error: MPS backend out of memory (MPS allocated: 5.77 GiB, other allocations: 36.91 MiB, max allowed: 8.29 GiB). Tried to

: 